# SCI1070 — Semaine 15 : caractériser un problème, et le dire honnêtement

| Application | Statut dans ce cours |
| --- | --- |
| **Le marché** | **déjà fait**, en semaine 7 |
| **Le paragraphe** | **s'exécute**, avec trois réserves (plateforme, versions, paiement) |
| **La page rendue** | **facultative, non notée, hors du carnet** |
| **La permutation** | **démonstration seulement** : six jours de calcul |

> **Pourquoi ce tableau est en tête du laboratoire.** Pendant treize semaines, ce
> cours a distingué ce qui s'exécute de ce qui se raconte. Une dernière semaine qui
> annoncerait quatre laboratoires exécutables et n'en livrerait aucun annulerait ce
> travail d'un coup. **C'est l'état réel des choses, et le savoir fait partie de ce
> que cette semaine enseigne.**

In [ ]:
import importlib.util
import subprocess
import sys

for paquet, module in [("gymnasium>=1.0", "gymnasium"), ("torch", "torch")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", paquet], check=True)
if importlib.util.find_spec("sci1070") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "git+https://github.com/FaycalZM/drl-course-teluq.git#subdirectory=code"],
                   check=True)

import numpy as np
import torch
import matplotlib.pyplot as plt

print("prêt |", sys.platform)

---
# Volet A : mesurer ce qu'une liste de commandes admissibles fait à un problème

**C'est le volet principal, le seul qui tourne de bout en bout.** Il ne contient
**aucun entraînement** : la semaine porte sur la **caractérisation d'un problème**.

> **Réserve de plateforme, et elle est réelle.** Le paquet ne prend en charge que
> Linux et macOS. Il s'installe dans Colab **parce que Colab tourne sous Linux** ;
> sur une machine Windows il demande un conteneur.

In [ ]:
if sys.platform.startswith("linux") or sys.platform == "darwin":
    if importlib.util.find_spec("textworld") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "textworld"],
                       check=True)
    import textworld
    import textworld.gym
    print("textworld disponible")
else:
    print(f"plateforme {sys.platform} : le paquet ne s'installe pas ici.")
    print("Ouvrez ce carnet dans Colab pour le volet A.")

**Étape 1 — engendrer un jeu.** Une quête de **longueur 5**, avec une graine fixée
**et notée**. **Sans votre protocole, votre mesure de l'étape 4 ne sera comparable à
rien.**

In [ ]:
...        # À COMPLÉTER : les cinq paramètres, notés avant de lancer, puis le jeu

**Étape 2 — écrire l'adaptateur.** L'environnement renvoie **quatre** valeurs au lieu
de cinq, et **ne déclare ni espace d'observations ni espace d'actions**.

In [ ]:
# Ma décision sur `tronque`, et pourquoi : je tiens MON PROPRE compteur de pas et
# je tronque à 100. Mettre `tronque` toujours à faux rendrait la boucle infinie
# sur un jeu que l'agent ne sait pas finir, et masquerait la distinction que la
# semaine 11 a passé son temps à défendre. Le non-choix n'en est pas un.
class AdaptateurTexte:
    """Enveloppe un environnement textuel pour qu'il ressemble à du Gymnasium."""

    def __init__(self, env, max_pas=100):
        ...        # À COMPLÉTER : l'environnement, le compteur, la limite

    def reset(self, seed=None, options=None):
        ...        # À COMPLÉTER : rendre (obs, info)

    def step(self, action):
        ...        # À COMPLÉTER : rendre CINQ valeurs

**Étape 4 — la mesure du volet.**

In [ ]:
# L'espace de production, pour avoir l'ordre de grandeur sous les yeux.
espace_production = 1250 ** 8
print(f"espace de production par pas : 1250^8 = {espace_production:.3e}")

for mediane in [5, 12, 30]:
    print(f"  médiane {mediane:3d} -> rapport ~ 10^{np.log10(espace_production / mediane):.1f}")

In [ ]:
...        # À COMPLÉTER : les vingt premiers états, l'histogramme, la médiane et le rapport

**Étape 3 — lesquels de ces champs un environnement industriel vous donnerait-il ?**
La description de la pièce et l'inventaire, oui : ce sont des observations. Le nom du
lieu, peut-être. Les faits du monde, les commandes admissibles et la suite d'actions
gagnante, **non** : ce sont des privilèges de simulateur, et un système réel n'a ni
la liste de ce qui est légal ni la solution.

**Étape 5 — l'autre béquille.** **Lequel des deux réglages change la classe du
problème, et lequel ne change que sa difficulté ?** La **liste admissible** change la
**classe** : sans elle, l'espace d'actions est combinatoire et aucune des méthodes du
cours ne s'applique telle quelle. La **récompense intermédiaire** ne change que la
**difficulté** : le problème reste le même, il devient seulement moins creux.

**Ce que ce volet établit** : que la réduction d'espace d'actions est **mesurable, et
de combien**. **Ce qu'il n'établit pas** : quoi que ce soit sur la performance d'un
agent, puisque vous n'en entraînez aucun.

---
# Volet B : lire une divergence, et poser l'ancre qui la répare

Volet de **démonstration**. Vous n'entraînez rien : vous recevez des journaux et vous
les lisez.

In [ ]:
from sci1070.donnees import charger_journal, journaux_disponibles

print("journaux fournis :", journaux_disponibles())

journal = charger_journal("cube-sans-ancre")
print(journal.description)
print(f"{len(journal.pas)} points, de {journal.pas[0]} à {journal.pas[-1]} pas de gradient")

**Étape 1 — le diagnostic.**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(journal.pas, journal.perte_politique, label="perte de politique")
ax.plot(journal.pas, journal.perte_valeur, label="perte de valeur")
ax.set_yscale("log")
ax.set_xlabel("pas de gradient"); ax.set_ylabel("perte")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title(journal.titre)
plt.show()

In [ ]:
...        # À COMPLÉTER : les trois réponses, lues sur les courbes

**Étape 2 — l'explication, sans code.** La cible de valeur de la section 15.5 définit
chaque valeur **par rapport à celles de ses voisins**. Rien n'y fixe d'échelle : si
toutes les valeurs montent ensemble, la cible monte avec elles et l'erreur reste
petite. **L'état résolu est le seul endroit de l'espace où une valeur est connue**,
parce que c'est le seul dont on sait qu'il vaut exactement zéro coup. **C'est le même
mécanisme que la semaine 14** : une perte qui ne dépend que de **différences** laisse
l'échelle dériver, et c'est l'invariance par décalage du volet A de cette semaine-là.

**Étape 3 — le correctif, comme fonction pure.**

In [ ]:
def cible_sans_ancre(valeurs_enfants, indicateurs_but, est_resolu):
    """La cible de la section 15.5. Aucune échelle n'y est fixée."""
    ...        # À COMPLÉTER : le meilleur enfant, et rien d'autre


def cible_avec_ancre(valeurs_enfants, indicateurs_but, est_resolu):
    """La cible de la section 15.7. L'état résolu ancre l'échelle."""
    ...        # À COMPLÉTER : la même chose, SAUF pour l'état résolu, dont la valeur est connue


cas = {
    "loin du but":        (torch.tensor([0.1, 0.2, 0.05]), torch.tensor([0., 0., 0.]), False),
    "à une rotation":     (torch.tensor([0.1, 0.9, 0.05]), torch.tensor([0., 1., 0.]), False),
    "l'état résolu":      (torch.tensor([0.3, 0.3, 0.30]), torch.tensor([0., 0., 0.]), True),
}

for nom, (v, g, resolu) in cas.items():
    a = float(cible_sans_ancre(v, g, resolu))
    b = float(cible_avec_ancre(v, g, resolu))
    marque = "DIFFÈRENT" if abs(a - b) > 1e-9 else "coïncident"
    print(f"{nom:18s} sans ancre {a:8.4f} | avec ancre {b:8.4f}   {marque}")

**Étape 4 — la lecture des mesures fournies.** Le correctif a amélioré la proportion
de cubes résolus, et surtout il a **supprimé la dérive de la fonction de valeur**.
**Et la profondeur 13 ne doit pas être comparée directement à la profondeur de
brouillage de 1 000 du protocole publié** : ce ne sont pas les mêmes grandeurs. Un
brouillage profond repasse souvent près de l'état résolu, donc sa difficulté réelle
n'est pas celle que son nombre annonce. **C'est la section 15.8 qui donne les mots :
un chiffre de banc d'essai sans son protocole n'est pas une mesure.**

In [ ]:
journal_avec = charger_journal("cube-avec-ancre")
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(journal.pas, journal.perte_valeur, label="sans ancre")
ax.plot(journal_avec.pas, journal_avec.perte_valeur, label="avec ancre")
ax.set_yscale("log"); ax.set_xlabel("pas de gradient"); ax.set_ylabel("perte de valeur")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

print("Rappel, et il est dans l'objet lui-même :")
print(journal_avec.description)

**Ce que ce volet établit** : que vous savez diagnostiquer une fonction de valeur
sans ancre et écrire le correctif. **Ce qu'il n'établit pas** : que le correctif
résout le problème, puisque la mesure fournie montre justement **qu'il ne le résout
pas sur le cube complet**.

---
# Volet C : facultatif et non noté, et il demande un navigateur

**Ce volet ne tourne pas dans un carnet, et il n'est pas noté.** Si l'une des quatre
conditions manque, passez-le : **il n'enlève rien à votre évaluation**.

In [ ]:
# À exécuter sur votre machine de bureau, pas ici.
#
# Étape 1 : émettez le MÊME clic correct deux fois, une fois avec le navigateur
#           dessiné à l'écran, une fois sans interface. Journalisez les TROIS
#           valeurs : récompense fournie, récompense brute, temps écoulé.
# Étape 2 : répétez chaque condition dix fois. Vous devez obtenir DEUX
#           POPULATIONS QUI NE SE RECOUVRENT PAS, pour une récompense brute
#           identique.
print("volet hors carnet : voir l'énoncé")

**Étape 3, la question du volet.** **Qu'est-ce qu'un score publié sur ce jeu de
référence mesure exactement ?** Il mesure la **justesse de l'action et la vitesse de
la machine**, mélangées en un seul nombre, parce que la récompense décroît avec le
temps écoulé. **Ce qu'il faudrait publier à côté pour qu'il soit comparable** : le
matériel, le mode de rendu (avec ou sans interface), et la récompense **brute**,
c'est-à-dire avant la décroissance temporelle.

---
# Volet D : la lecture, et elle ne demande ni machine ni paiement

**Ce que je vous demande**, et c'est un exercice d'**écriture** :

1. Un paragraphe de dix lignes qui rapporte le résultat **honnêtement** : le chiffre,
   le protocole, les **quatre réserves** de la section 15.9, et la conclusion que le
   résultat autorise.
2. Le même paragraphe **malhonnêtement** : **en gardant tous les chiffres exacts**
   mais en omettant ce qu'il faut pour que la conclusion paraisse plus large qu'elle
   n'est.

Mettez les deux côte à côte et **gardez-les**. C'est le meilleur entraînement que je
connaisse à la relecture de son propre rapport, et **c'est le dernier exercice que ce
cours vous propose**.

> **Et pour que l'exercice soit complet, voici le même piège une dernière fois, dans
> ce carnet.** Les journaux du volet B portent dans leur champ `description` la
> mention qu'ils sont **reconstruits et non mesurés**. Un rapport qui tracerait ces
> courbes en les présentant comme une mesure aurait tous ses chiffres exacts et
> serait malhonnête. **C'est exactement le paragraphe numéro deux que je vous demande
> d'écrire.**

---
## Pas de solution courte cette semaine, et c'est délibéré

**Aucune bibliothèque ne caractérise un problème à votre place.** Il n'existe pas de
fonction qui vous dise de quelle nature est votre action, qui a écrit votre
récompense, et laquelle des quatorze semaines s'applique. **C'est le seul travail de
ce cours qui ne s'automatise pas, et c'est pour cela qu'il est la dernière chose
qu'il vous apprend.**